# Figure 4 — cell-type FGES comparison plots, **validation cohort** (CHESS-1336)

Reproduces the four published `Cell_type_FGES_comparison` figure families
(`Box_plot`, `Median_heatmaps`, `Scater_plots`, `Stat_fges_table`) — but sourced
from the **validation** output of `mapping_ssgseas/Signatures comparison VALIDATION.ipynb`
(`plots/mapping_ssgseas_validation.pkl`) instead of the published
`data/mapping_ssgseas.pkl`.

All artefacts carry a `_validation` suffix; the published SVGs / tables are never
overwritten.

**Structure difference to keep in mind:** validation ssGSEA frames are indexed by
plain sample ids (SRX...), not the `('CellType','run')_...` composite labels of v1.
The run-selection / bootstrap variants of the published `Box_plot` notebook are
therefore out of scope here (there are no "runs"); the per-source **violins** are
reproduced via `plot_violin_per_source`. Median heatmaps, F1-vs-CV scatter and the
stats table share one brick - `fges_metrics` (section M) - computed straight on the
validation cohort by passing `validation_annot` / `validation_expr`-derived ranks
as keyword arguments to `get_strat_cell_type` / `derive_rank_deviation`.

Run on an environment with S3 access (JupyterHub): the setup section pulls
`validation_expr` from the osrp zarr.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config IPCompleter.use_jedi = False

In [ ]:
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
import seaborn as sns
import zarr
from loguru import logger
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

import signature_validation.plotting.plotting as pl_mod
from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
)
from signature_validation.benchmark.plotting import plot_violin_per_source
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import (
    boxplot_with_pvalue,
    calculate_and_plot_correlations,
    default_cmap,
    plot_scatter_with_ci,
    plot_scatter_with_ci_agg,
)
from signature_validation.ssgsea_calc.ssgsea_calc import detect_fges_source
from signature_validation.utils.fges_utils import (
    derive_rank_deviation,
    get_metric_for_signature,
    get_strat_cell_type,
)
from signature_validation.utils.utils import (
    df_fisher_chi2,
    median_scale,
    print_95ci_of_mean,
    scale_series,
    to_common_samples,
)

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'png'
plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 120
sns.set_style("ticks")

In [ ]:
import os

# cwd -> Cell_type_FGES_comparison/ (this notebook's folder), so the relative
# ./plots ./tables ./data paths match the published notebooks.
nb = next(Path.home().rglob("Figure4_plots_VALIDATION.ipynb"), None)
if nb:
    os.chdir(nb.parent)
print("cwd ->", os.getcwd())

## Setup - inputs (re-derive light objects, load the heavy precomputed scores)

Light objects (annotation, mapping, controls, harmonized gmt) are cheap to rebuild
and mirror sections 3-6 of the main validation notebook. The heavy `mapping_ssgseas`
is **loaded** from `plots/mapping_ssgseas_validation.pkl` (produced by section 7
there) - we never recompute ssGSEA here.

In [ ]:
DATA_DIR = Path("./data")
PLOTS_DIR = Path("./plots")
TABLES_DIR = Path("./tables")

VALIDATION_ANNOT_PATH = DATA_DIR / "sorted_cells_to_check_all_annot.tsv"
V1_GMT_PICKLE = DATA_DIR / "msigdb_gmt.pkl"
MAPPING_SSGSEAS_PATH = PLOTS_DIR / "mapping_ssgseas_validation.pkl"

# osrp zarr v3 (obs x var): SRX samples x gene symbols. See main validation nb section 4.
OSRP_ZARR = (
    "bostongene-eurynome-exchange/raw_data/v2/expressions/osrp_tier1_expressions.zarr"
)

# Shared metrics brick (this notebook writes it, section M).
FGES_METRICS_PATH = DATA_DIR / "fges_metrics_controls_goi_wo_bootstrap_validation.pkl"

for label, pth in (
    ("validation annotation", VALIDATION_ANNOT_PATH),
    ("v1 GMT pickle", V1_GMT_PICKLE),
    ("validation mapping_ssgseas", MAPPING_SSGSEAS_PATH),
):
    if not pth.exists():
        logger.error("{} not found: {}", label, pth)

In [ ]:
validation_annot = load_new_cohort_annotation(VALIDATION_ANNOT_PATH)

# Explicit OD-128 QC filter (identical to the main validation notebook section 3):
# Technical_QC == True AND Decision_deconvolution_without_parent != "False"
# (compare to the STRING "False"; NaN -> 'nan' correctly passes as "Kassandra
# made no prediction", not a rejection).
qc_mask = (validation_annot["Technical_QC"] == True) & (
    validation_annot["Decision_deconvolution_without_parent"].astype(str) != "False"
)
validation_annot = validation_annot[qc_mask]
logger.info("annotation after QC filter: {} samples", len(validation_annot))
validation_annot["Cell_type"].value_counts()

In [ ]:
def load_osrp_expressions(
    sample_ids: list[str],
    zarr_path: str = OSRP_ZARR,
    log2: bool = True,
) -> pd.DataFrame:
    """Load osrp zarr expressions as a genes x samples frame (see main nb section 4).

    Parameters
    ----------
    sample_ids : list[str]
        Sample ids from the annotation index; intersection with ``obs_names`` is
        taken automatically.
    zarr_path : str
        Bucket+key of the zarr array (no ``s3://`` scheme).
    log2 : bool
        Apply ``log2(TPM + 1)`` (v1-pipeline transform).

    Returns
    -------
    pd.DataFrame
        Genes x samples (index = ``var_names``, columns = matched ``obs_names``).

    Raises
    ------
    KeyError
        If no ``sample_ids`` are present in the zarr ``obs_names``.
    """
    fs = s3fs.S3FileSystem()
    store = zarr.storage.FsspecStore(fs, path=zarr_path)
    z = zarr.open(store, mode="r")

    obs = list(z.attrs["obs_names"])
    var = list(z.attrs["var_names"])
    pos = {s: i for i, s in enumerate(obs)}

    want = [s for s in sample_ids if s in pos]
    if not want:
        raise KeyError("no sample_ids found in osrp zarr obs_names")
    rows = sorted(pos[s] for s in want)
    logger.info("osrp zarr: {} / {} samples matched", len(rows), len(sample_ids))

    x = z.oindex[rows, :]
    expr = pd.DataFrame(x.T, index=var, columns=[obs[r] for r in rows])
    if log2:
        expr = np.log2(expr + 1)
    logger.info("expressions: {} genes x {} samples", expr.shape[0], expr.shape[1])
    return expr


validation_expr = load_osrp_expressions(list(validation_annot.index))
validation_expr.shape

In [ ]:
mapping = build_mapping(annotation=validation_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, validation_annot)
logger.info(
    "in-scope FGES: {}; controls present: {}", len(mapping), len(controls_present)
)

In [ ]:
# Reuse v1 gene lists byte-identical, then harmonize to the validation gene index.
v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)
for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    assert count_random_fges(v1_gmt[sign]) == 10, f"{sign}: expected 10 RANDOM_FGES"
msigdb_gmt = harmonize_gmt_to_index(v1_gmt, validation_expr.index)
logger.info(
    "msigdb_gmt: {} FGES, {} sub-signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

In [ ]:
with open(MAPPING_SSGSEAS_PATH, "rb") as handle:
    mapping_ssgseas = pickle.load(handle)
for key in mapping_ssgseas.keys():
    for ct in mapping_ssgseas[key]["Goi"].keys():
        print(key, "\t", ct, "\t", mapping_ssgseas[key]["Goi"][ct].shape)

# Section B - Box_plot family: per-source violins

Reuses `plot_violin_per_source` (the modernized v1 cells 60-68). GOI and Control
violins per FGES source, non-scaled ssGSEA. Rare cell types / rare FGES are starred
automatically (none in-scope here). Writes
`plots/violin_comparison_nonscaled_validation_{GOI,control}.svg`.

In [ ]:
plot_violin_per_source(mapping_ssgseas, save_dir=PLOTS_DIR, suffix="_validation")
logger.info("violins saved under {}", PLOTS_DIR)

# Section M - `fges_metrics` on the validation cohort (shared brick)

Port of `Scater_plots` cell 20: for every FGES x sub-signature, `n_iter` stratified
bootstrap rounds of GOI-vs-Control classification metrics (F1, Accuracy, ROC/PR-AUC)
plus rank-deviation CVs (`goi_cv`, `control_cv`). The only change vs v1 is that cell
types come from `validation_annot` and ranks from `validation_expr`, passed as
keyword arguments - so the helpers work on the plain-sample-id validation frames.

First, the v1 daughter-column cleaning (cell 16) and the `Th2_cells` drop (cell 19),
both guarded for what is actually present in the validation cohort.

In [ ]:
# v1 cell 16 - remove daughter-signature columns from parent GOI frames so a
# parent FGES is not credited with its daughters' signatures. Guarded to keys
# present in the in-scope validation mapping.
parent_to_daughter = {
    "Main4_T_cells": [
        "Main4_Th17_signature",
        "Main4_Th1_signature",
        "Main4_CD8_T_cells",
        "Main4_Treg",
        "Main4_Effector_cells",
        "Main4_CD4_T_cells",
        "Main4_Follicular_helper_T_cells",
    ],
    "Main4_CD4_T_cells": [
        "Main4_Th17_signature",
        "Main4_Th1_signature",
        "Main4_Follicular_helper_T_cells",
        "Main4_Treg",
    ],
    "Main4_B_cells": ["Main4_Plasma_cells"],
    "Main4_Pan_macrophage_signature": ["Main4_M2_signature"],
    "Main4_Monocyte": ["Main4_Pan_macrophage_signature", "Main4_M2_signature"],
    "Main4_Endothelium": ["Main4_Lymphatic_endothelium"],
}
for parent, daughters in parent_to_daughter.items():
    if parent not in mapping_ssgseas:
        continue
    daughter_columns = set()
    for daughter in daughters:
        if daughter not in mapping_ssgseas:
            continue
        for cell_type in mapping_ssgseas[daughter]["Goi"]:
            daughter_columns.update(mapping_ssgseas[daughter]["Goi"][cell_type].columns)
    for cell_type, frame in mapping_ssgseas[parent]["Goi"].items():
        clean_cols = [c for c in frame.columns if c not in daughter_columns]
        mapping_ssgseas[parent]["Goi"][cell_type] = frame[clean_cols]

In [ ]:
# v1 cell 19 - Th2 has too few sorted samples to be a reliable control.
validation_annot = validation_annot[validation_annot.Cell_type != "Th2_cells"]

# Rank matrix + gene universe for derive_rank_deviation (validation-cohort versions
# of Scater_plots cell 15's ranked_expr / pipeline_genes).
ranked_expr = validation_expr.rank(pct=True)
pipeline_genes = validation_expr.index.to_list()

In [ ]:
fges_metrics = {}
n_iter = 10
total = sum(len(msigdb_gmt[i].keys()) for i in msigdb_gmt.keys())
pbar = tqdm(total=total * n_iter, desc="Processing", position=0, leave=True)
for sign in list(mapping_ssgseas.keys())[::-1]:
    control = pd.concat(mapping_ssgseas[sign]["Control"].values())
    control = control[~control.index.duplicated()]
    goi = pd.concat(mapping_ssgseas[sign]["Goi"].values())
    goi = goi[~goi.index.duplicated()]
    for fges in goi.columns:
        fges_metrics[fges] = {}
        for seed in range(n_iter):
            new_control, cell_types = get_strat_cell_type(
                control, seed, public_cells_annot=validation_annot
            )
            if len(goi) > len(new_control) and len(goi) > 50:
                goi_size = len(new_control)
            elif len(goi) < 50:
                goi_size = 50
            else:
                goi_size = len(goi)
            if len(goi) <= 100 or len(new_control) <= 100:
                control_size = 100
            elif len(goi) > len(new_control) and len(new_control) > 100:
                control_size = len(new_control)
            else:
                control_size = len(goi)
            sample_perc = control_size / (len(new_control) + len(goi))
            _, control_subsample = train_test_split(
                new_control,
                test_size=sample_perc,
                stratify=cell_types,
                random_state=seed,
            )
            goi_subsample = goi.sample(n=goi_size, replace=True, random_state=seed)
            labels = pd.concat(
                [
                    pd.Series(index=control_subsample.index, data=0),
                    pd.Series(index=goi_subsample.index, data=1),
                ]
            )
            df = pd.concat([control_subsample, goi_subsample])
            m_dict = get_metric_for_signature(
                df[fges], labels, verbose=False, youden_thr=False
            )
            dev_dict = derive_rank_deviation(
                control_subsample,
                goi,
                sign,
                fges,
                msigdb_gmt=msigdb_gmt,
                ranked_expr=ranked_expr,
                pipeline_genes=pipeline_genes,
            )
            m_dict.update(dev_dict)
            fges_metrics[fges][seed] = m_dict
            pbar.update(1)
pbar.close()

In [ ]:
with open(FGES_METRICS_PATH, "wb") as handle:
    pickle.dump(fges_metrics, handle, pickle.HIGHEST_PROTOCOL)
logger.info("wrote {} ({} sub-signatures)", FGES_METRICS_PATH, len(fges_metrics))

# Section H - Median heatmaps (top-10 per FGES)

Port of `Median_heatmaps` cells 11/13/16-17/21. Per BG signature, the top-10
sub-signatures (ranked by `goi_cv` asc, `F1` desc, `control_cv` asc from section M)
are shown as a median-scaled cell-type heatmap. Display cell-type order =
`controls_present` (validation-scoped); samples of other types fall into
`Other_controls`. Writes `plots/svg_pictures_median_heatmaps/*_validation.svg` and
the BG-signs summary `plots/top10_median_heatmap_for_bg_signs_validation.svg`.

In [ ]:
# Wide sample x sub-signature frame (all groups, all FGES), deduplicated (v1 cell 11).
_frames = []
for sign in mapping_ssgseas.keys():
    control_and_goi = []
    for group in mapping_ssgseas[sign].keys():
        for frame in mapping_ssgseas[sign][group].values():
            control_and_goi.append(frame)
    control_and_goi = pd.concat(control_and_goi)
    control_and_goi = control_and_goi[~control_and_goi.index.duplicated(keep="first")]
    _frames.append(control_and_goi)
df_wide = pd.concat(_frames, axis=1)
df_wide = df_wide.loc[:, ~df_wide.columns.duplicated()]

# Cell-type labels per sample; non-displayed types -> Other_controls (v1 cell 13).
controls_order_short = list(controls_present)
labels = validation_annot["Cell_type"].copy()
labels = pd.concat(
    [
        labels[labels.isin(controls_order_short)],
        pd.Series(
            index=labels[~labels.isin(controls_order_short)].index,
            data="Other_controls",
        ),
    ]
)
if (labels == "Other_controls").any() and "Other_controls" not in controls_order_short:
    controls_order_short = controls_order_short + ["Other_controls"]

In [ ]:
def _median_heatmap_for_bg(
    bg_sign, top, clip=None, out_subdir="svg_pictures_median_heatmaps", prefix="top10"
):
    """Build one top-`top` median-scaled heatmap for a BG signature."""
    signs = msigdb_gmt[bg_sign].keys()
    scores = pd.DataFrame(
        {s: pd.DataFrame(fges_metrics[s]).T.mean() for s in signs if s in fges_metrics}
    ).T
    ind = scores.sort_values(
        by=["goi_cv", "F1", "control_cv"], ascending=[True, False, True]
    ).index
    ind = ind[~ind.duplicated()].to_list()
    ind = [i for i in ind if "RANDOM" not in i][:top]
    if bg_sign not in ind:
        ind = ind[: (top - 1)] + [bg_sign]
    ind = [i for i in ind if i in df_wide.columns]
    data = df_wide[ind].T
    data = data[~data.index.duplicated()]
    ls = labels.reindex(data.columns)
    cols = [c for c in controls_order_short if c in set(ls.dropna())]
    data = data.T.groupby(ls).mean().T[cols]

    scaled = median_scale(data.T).T
    if clip is not None:
        scaled = scaled.clip(-clip, clip)

    fig, ax = plt.subplots(figsize=(25, 10))
    sns.heatmap(
        scaled,
        cmap=default_cmap,
        xticklabels=True,
        yticklabels=True,
        ax=ax,
        cbar=True,
        annot=False,
        fmt=".1f",
        square=True,
        cbar_kws={"shrink": 0.5},
        linewidths=0.5,
    )
    ax.set_xticklabels([c.replace("_", " ") for c in cols], rotation=90)
    ax.set_yticklabels(ind)
    plt.tight_layout(pad=0.2)
    out = PLOTS_DIR / out_subdir
    out.mkdir(parents=True, exist_ok=True)
    plt.savefig(
        out / f"{prefix}_median_heatmap_for_{bg_sign}_validation.svg", format="svg"
    )
    plt.close(fig)


plt.rcParams["svg.fonttype"] = "none"
for bg_sign in msigdb_gmt.keys():
    _median_heatmap_for_bg(bg_sign, top=10, clip=None, prefix="top10")
    _median_heatmap_for_bg(bg_sign, top=10, clip=3.5, prefix="scaled_top10")
logger.info(
    "median heatmaps saved under {}", PLOTS_DIR / "svg_pictures_median_heatmaps"
)

In [ ]:
# BG-signs-only summary heatmap (v1 cell 21), restricted to in-scope BG signatures.
plt.rcParams["svg.fonttype"] = "none"
bg_ind_full = [
    "Main4_T_cells",
    "Main4_CD4_T_cells",
    "Main4_Th1_signature",
    "Main4_Th17_signature",
    "Main4_Follicular_helper_T_cells",
    "Main4_Treg",
    "Main4_Effector_cells",
    "Main4_CD8_T_cells",
    "Main4_NK_cells",
    "Main4_B_cells",
    "Main4_Plasma_cells",
    "Main4_Neutrophil_signature",
    "Main4_Eosinophil_signature",
    "Main4_Mast_cell_signature",
    "Main4_Monocyte",
    "Main4_Pan_macrophage_signature",
    "Main4_M2_signature",
    "Main4_Endothelium",
    "Main4_Lymphatic_endothelium",
]
bg_ind = [i for i in bg_ind_full if i in df_wide.columns]
data = df_wide[bg_ind].T
ls = labels.reindex(data.columns)
cols = [c for c in controls_order_short if c in set(ls.dropna())]
data = data.T.groupby(ls).mean().T[cols]
fig, ax = plt.subplots(figsize=(25, 10))
sns.heatmap(
    data,
    cmap=default_cmap,
    xticklabels=True,
    yticklabels=True,
    ax=ax,
    cbar=True,
    annot=False,
    fmt=".1f",
    square=True,
    cbar_kws={"shrink": 0.5},
    linewidths=0.5,
)
ax.set_xticklabels([c.replace("_", " ") for c in cols], rotation=90)
ax.set_yticklabels(bg_ind)
plt.tight_layout(pad=0.2)
plt.savefig(PLOTS_DIR / "top10_median_heatmap_for_bg_signs_validation.svg", format="svg")
plt.close(fig)

# Section S - Scatter: weighted F1 vs GOI rank-CV

Port of `Scater_plots` cells 23/25. Per-FGES scatter (`plot_scatter_with_ci`) and the
across-FGES aggregate (`plot_scatter_with_ci_agg`), colored by FGES source.

The module-level `signature_palette` used inside those functions is a list in the
current package; we bind the source->color **dict** onto the module so the functions
resolve colors by source name (v1 behavior).

In [ ]:
signature_palette = {
    "Internal": "#ff0000",
    "Random_FGES": "black",
    "xCell": "#12ff1b",
    "Bindea": "#ff8000",
    "Nirmal": "#804080",
    "Gene_Ontology": "#40fbc9",
    "KEGG": "gold",
    "BioCarta": "#fe7cc3",
    "WikiPathways": "#0080ff",
    "Reactome": "#258103",
    "Pathway_Interaction_Database": "#83e9ff",
    "Human_Phenotype_Ontology": "#0500f5",
    "MSigDb_Dif_Expression": "#93ae8b",
    "MSigDb_Single_Cell": "#df7ffe",
    "MSigDb_Other": "#008080",
}
# plot_scatter_with_ci / _agg read plotting.signature_palette[source] internally.
pl_mod.signature_palette = signature_palette

In [ ]:
scatter_dir = PLOTS_DIR / "svg_pictures_F1_cv_validation"
scatter_dir.mkdir(parents=True, exist_ok=True)
for bg_sign in msigdb_gmt.keys():
    alt_signs = msigdb_gmt[bg_sign].keys()
    plot_dict = {i: fges_metrics[i] for i in alt_signs if i in fges_metrics}
    plot_scatter_with_ci(
        plot_dict,
        title=f"Comparison for {bg_sign}_validation",
        path=scatter_dir,
        xlabel="Coeffient of variation of ranks in GOI",
        ylabel="Weighted F1-score (GOI vs Controls)",
    )

In [ ]:
plot_scatter_with_ci_agg(
    fges_metrics,
    title="Comparison for FGES types_validation",
    path=PLOTS_DIR,
    xlabel="Coeffient of variation of ranks in GOI",
    ylabel="Weighted F1-score (GOI vs Controls)",
)

# Section T - Stats table

Port of `Stat_fges_table` cells 14-25: per (BG FGES x sub-signature) means, paired
Wilcoxon of BG vs each sub-signature (GOI and Control), the section-M classification
metrics, and FGES source. FDR-corrected, sorted, written to
`tables/fges_stats_table_validation.tsv`. A few summary views follow (source-level
F1 / CV, the per-source F/CV heatmaps, and the CV<->F correlation).

In [ ]:
out_rows = []
for bg_sign in msigdb_gmt.keys():
    goi = pd.concat(mapping_ssgseas[bg_sign]["Goi"].values())
    control = pd.concat(mapping_ssgseas[bg_sign]["Control"].values())
    signs = msigdb_gmt[bg_sign].keys()
    scores = pd.DataFrame(
        {s: pd.DataFrame(fges_metrics[s]).T.mean() for s in signs if s in fges_metrics}
    ).T
    ind = scores.sort_values(
        by=["goi_cv", "F1", "control_cv"], ascending=[True, False, True]
    ).index
    goi_pv, control_pv = {}, {}
    for comp_s in ind:
        if bg_sign != comp_s:
            x, y = to_common_samples((goi[bg_sign].dropna(), goi[comp_s].dropna()))
            _, goi_pv[comp_s] = wilcoxon(x, y)
            x, y = to_common_samples(
                (control[bg_sign].dropna(), control[comp_s].dropna())
            )
            _, control_pv[comp_s] = wilcoxon(x, y)
        else:
            goi_pv[comp_s] = 1
            control_pv[comp_s] = 1
    o = pd.DataFrame(
        {
            "GOI_Mean": goi[ind].mean(),
            "GOI_p_adj_paired_wilcoxon": pd.Series(goi_pv),
            "Control_Mean": control[ind].mean(),
            "Control_p_adj_paired_wilcoxon": pd.Series(control_pv),
            "F_score": scores["F1"].loc[ind],
            "Accuracy": scores["Accuracy"].loc[ind],
            "PR_AUC": scores["PR_AUC"].loc[ind],
            "ROC_AUC": scores["ROC_AUC"].loc[ind],
            "GOI_cv": scores["goi_cv"].loc[ind],
            "Control_cv": scores["control_cv"].loc[ind],
        }
    ).loc[ind]
    o["BG_FGES"] = bg_sign
    o["Alt_FGES"] = ind
    o["FGES_source"] = o.Alt_FGES.map(detect_fges_source)
    o["Genes"] = pd.Series(
        {k: ", ".join(list(msigdb_gmt[bg_sign][k].genes)) for k in signs}
    )
    out_rows.append(o)
out = pd.concat(out_rows)
out.FGES_source.value_counts()

In [ ]:
for col in ["GOI_p_adj_paired_wilcoxon", "Control_p_adj_paired_wilcoxon"]:
    _, corr, _, _ = multipletests(out[col], method="fdr_bh")
    out[col] = pd.Series(data=corr, index=out.index)
out = out.sort_values(by=["BG_FGES", "F_score"], ascending=[True, False])
out.to_csv(TABLES_DIR / "fges_stats_table_validation.tsv", sep="\t", index=False)
logger.info(
    "wrote {} ({} rows)", TABLES_DIR / "fges_stats_table_validation.tsv", len(out)
)
out.head()

## Section T - summary views (for discussion)

In [ ]:
out = out[~out.index.duplicated()]
print("Mean F-score by source:")
print(out.F_score.groupby(out.FGES_source).mean().sort_values(ascending=False))
print("\nMean GOI_cv by source:")
print(out.GOI_cv.groupby(out.FGES_source).mean().sort_values(ascending=True))

In [ ]:
# Per-source F_score / GOI_cv heatmap per BG FGES (v1 cell 91), for sources present.
all_stats_by_ct = {typ: {} for typ in out.FGES_source.unique()}
for sign in out.BG_FGES.unique():
    part = out[out.BG_FGES == sign]
    for typ in part.FGES_source.unique():
        all_stats_by_ct[typ][sign] = part[part.FGES_source == typ][
            ["F_score", "GOI_cv"]
        ].mean()

for typ, per_sign in all_stats_by_ct.items():
    if not per_sign:
        continue
    x = pd.DataFrame(per_sign).T[["F_score", "GOI_cv"]]
    fig, ax = plt.subplots(figsize=(6, 8))
    sns.heatmap(x, annot=x, fmt=".2f", cmap=default_cmap, ax=ax, vmin=0, vmax=1)
    ax.set_title(typ)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# CV <-> F correlation, one representative sub-signature per BG FGES, bootstrapped
# (v1 cells 66-69). Excludes Random_FGES.
out_no_random = out[out.FGES_source != "Random_FGES"]
corrs = {}
for i in range(1000):
    samps = [
        group.sample(1, random_state=i).index[0]
        for _, group in out_no_random.groupby("BG_FGES")
    ]
    corrs[i] = calculate_and_plot_correlations(
        out_no_random.loc[samps].GOI_cv,
        out_no_random.loc[samps].F_score,
        name1="CV",
        name2="F",
        ret=True,
        plot=False,
        verbose=False,
    )
corrs = pd.DataFrame(corrs)
print("Spearman CV<->F across 1000 one-per-FGES resamples:")
print_95ci_of_mean(corrs.loc["Spearman"])

---
**Reasoning summary.** This notebook reuses the precomputed validation
`mapping_ssgseas` and rebuilds only the cheap cohort objects, then reproduces the
four published figure families with a `_validation` suffix. The single new
computation is the validation `fges_metrics` (section M), obtained by feeding
`validation_annot` / `validation_expr` ranks as keyword arguments to the existing
strat / rank-deviation helpers - which is what makes the median-heatmap,
F1-vs-CV scatter and stats-table ports work on plain-sample-id validation frames.